In [ ]:
# %%
# =========================================================
# IMPORT THƯ VIỆN
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# CÀI ĐẶT HIỂN THỊ
# =========================================================

pd.set_option('display.max_columns', None)

# %%
# =========================================================
# ĐỌC DATA GỐC
# =========================================================

df = pd.read_excel(
    r'C:\Users\T480s\Documents\COLLECT_DATA\data_raw.xlsx'
)

# %%
# =========================================================
# KIỂM TRA TỔNG QUAN DATASET
# =========================================================

df.info()

display(df.head())

print(f'Số dòng: {df.shape[0]}')
print(f'Số cột : {df.shape[1]}')

# %%
# =========================================================
# CHUẨN HÓA KIỂU DỮ LIỆU
# =========================================================

# ----- Chuyển disbursed_date về datetime -----

df['disbursed_date'] = pd.to_datetime(
    df['disbursed_date'],
    errors='coerce'
)

# ----- Tạo cột month -----

df['month'] = df['disbursed_date'].dt.month

# %%
# =========================================================
# XÓA CÁC CỘT KHÔNG SỬ DỤNG
# =========================================================

drop_cols = [

    'medium',
    'customer_created_at',
    'log_step_created_at',

    'product_code',
    'partner_score_band',
    'customer_segment',

    'last_medium',
    'is_new_customer',

    'created_year',
    'created_month',
    'created_date',

    'created_hour',
    'created_week',

    'segment',
    'nfc_status',
    'utm_source',

    'status_name',
    'month_name'

]

df = df.drop(
    columns=drop_cols,
    errors='ignore'
)

# %%
# =========================================================
# MAPPING TỈNH / THÀNH PHỐ -> VÙNG MIỀN
# =========================================================

north = {

    'Hòa Bình','Sơn La','Điện Biên','Lai Châu',
    'Lào Cai','Yên Bái','Phú Thọ','Hà Giang',
    'Tuyên Quang','Cao Bằng','Bắc Kạn',
    'Thái Nguyên','Lạng Sơn','Bắc Giang',
    'Quảng Ninh',

    'Hà Nội','Bắc Ninh','Hà Nam',
    'Hải Dương','Hải Phòng','Hưng Yên',
    'Nam Định','Thái Bình',
    'Vĩnh Phúc','Ninh Bình'

}

central = {

    'Thanh Hóa','Nghệ An','Hà Tĩnh',
    'Quảng Bình','Quảng Trị',
    'Thừa Thiên Huế',

    'Đà Nẵng','Quảng Nam',
    'Quảng Ngãi','Bình Định',
    'Phú Yên','Khánh Hòa',

    'Ninh Thuận','Bình Thuận',

    'Kon Tum','Gia Lai',
    'Đắk Lắk','Đắk Nông',
    'Lâm Đồng'

}

south = {

    'Hồ Chí Minh',
    'Bà Rịa - Vũng Tàu',
    'Bình Dương',
    'Bình Phước',
    'Đồng Nai',
    'Tây Ninh',

    'An Giang',
    'Bạc Liêu',
    'Bến Tre',
    'Cà Mau',
    'Cần Thơ',

    'Đồng Tháp',
    'Hậu Giang',
    'Kiên Giang',
    'Long An',

    'Sóc Trăng',
    'Tiền Giang',
    'Trà Vinh',
    'Vĩnh Long'

}

# =========================================================
# HÀM MAP REGION
# =========================================================

def map_region(x):

    if x == 'Hà Nội':
        return 'HN'

    elif x == 'Hồ Chí Minh':
        return 'HCM'

    elif x in north:
        return 'North'

    elif x in central:
        return 'Central'

    elif x in south:
        return 'South'

    else:
        return 'Other'

# ----- Province region -----

df['province_region'] = (
    df['province']
    .apply(map_region)
)

# ----- Current province region -----

df['current_province_region'] = (
    df['current_province']
    .apply(map_region)
)

# ----- Chuyển category -----

df['province_region'] = (
    df['province_region']
    .astype('category')
)

df['current_province_region'] = (
    df['current_province_region']
    .astype('category')
)

# %%
# =========================================================
# PHÂN NHÓM BIẾN
# =========================================================

# ---------------------------------------------------------
# NUMERICAL CONTINUOUS
# ---------------------------------------------------------

num_cont = [

    'loan_amount',
    'interestrate',
    'age',

    'score',
    'vc_score',
    'vc_pre_score',

    'emi',
    'emi_total',

    'dti',
    'dti_total',

    'residual_income',
    'approve_limit'

]

# ---------------------------------------------------------
# NUMERICAL DISCRETE
# ---------------------------------------------------------

num_disc = [

    'term',
    'month'

]

# ---------------------------------------------------------
# CATEGORICAL NOMINAL
# ---------------------------------------------------------

cat_nom = [

    'gender',
    'customer_type',

    'province_region',
    'current_province_region'

]

# ---------------------------------------------------------
# CATEGORICAL ORDINAL
# ---------------------------------------------------------

cat_ord = [

    'age_bin',
    'score_bin',
    'vc_score_bin',

    'loan_amount_bin',

    'emi_bin',
    'emi_total_bin',

    'dti_bin',
    'dti_total_bin',

    'residual_income_bin',
    'approve_limit_bin',

    'rank_group'

]

# %%
# =========================================================
# THIẾT LẬP THỨ TỰ CATEGORY
# =========================================================

rank_order = [

    'A1',
    'A2',
    'B',
    'C',
    'D',
    'E'

]

df['rank_group'] = pd.Categorical(

    df['rank_group'],

    categories=rank_order,

    ordered=True

)

# %%
# =========================================================
# KIỂM TRA MISSING VALUE
# =========================================================

def kiem_tra_missing(df):

    missing_summary = pd.DataFrame({

        'variable': df.columns,

        'missing_count': (
            df.isnull()
            .sum()
            .values
        ),

        'missing_percent (%)': (

            df.isnull()
            .mean()
            .values

            * 100

        )

    })

    # ----- Làm tròn -----

    missing_summary['missing_percent (%)'] = (

        missing_summary['missing_percent (%)']
        .round(2)

    )

    # ----- Sort giảm dần -----

    missing_summary = (

        missing_summary

        .sort_values(
            by='missing_count',
            ascending=False
        )

        .reset_index(drop=True)

    )

    return missing_summary

# ----- Hiển thị bảng missing -----

display(
    kiem_tra_missing(df)
)

# %%
# =========================================================
# KIỂM TRA DUPLICATE VALUE
# =========================================================

def kiem_tra_duplicate(df):

    duplicate_count = (
        df.duplicated()
        .sum()
    )

    duplicate_percent = (

        duplicate_count
        / len(df)

        * 100

    )

    print(
        f'Số dòng trùng lặp: {duplicate_count}'
    )

    print(
        f'Tỷ lệ trùng lặp: '
        f'{duplicate_percent:.2f}%'
    )

# ----- Chạy kiểm tra -----

kiem_tra_duplicate(df)

# %%
# =========================================================
# HÀM THỐNG KÊ MÔ TẢ
# DÀNH CHO NUMERICAL CONTINUOUS
# =========================================================

def thong_ke_mo_ta_num_cont(df, cols):

    # -----------------------------------------------------
    # TẠO PHÂN VỊ 10%
    # -----------------------------------------------------

    quantiles = np.arange(
        0,
        1.01,
        0.1
    )

    # -----------------------------------------------------
    # THỐNG KÊ MÔ TẢ
    # -----------------------------------------------------

    desc = (

        df[cols]

        .describe(
            percentiles=quantiles
        )

        .T

    )

    # -----------------------------------------------------
    # ĐỔI TÊN 50% -> MEDIAN
    # -----------------------------------------------------

    desc = desc.rename(
        columns={'50%': 'median'}
    )

    # -----------------------------------------------------
    # SẮP XẾP THỨ TỰ CỘT
    # -----------------------------------------------------

    col_order = (

        [

            'count',
            'mean',
            'median',
            'std',
            'min'

        ]

        +

        [

            f'{int(q*100)}%'

            for q in quantiles

            if q not in [0, 0.5, 1]

        ]

        +

        ['max']

    )

    desc = (

        desc[col_order]
        .round(2)

    )

    return desc

# %%
# =========================================================
# HÀM VẼ HISTOGRAM + BOXPLOT
# DÀNH CHO NUMERICAL CONTINUOUS
# =========================================================

def ve_hist_boxplot(
    df,
    col,
    figsize=(12,4)
):

    fig, axes = plt.subplots(
        1,
        2,
        figsize=figsize
    )

    # -----------------------------------------------------
    # HISTOGRAM
    # -----------------------------------------------------

    sns.histplot(
        data=df,
        x=col,
        kde=True,
        ax=axes[0]
    )

    axes[0].set_title(
        f'{col} - Histogram'
    )

    # -----------------------------------------------------
    # BOXPLOT
    # -----------------------------------------------------

    sns.boxplot(
        data=df,
        x=col,
        ax=axes[1]
    )

    axes[1].set_title(
        f'{col} - Boxplot'
    )

    plt.tight_layout()

    plt.show()

# %%
# =========================================================
# HÀM XỬ LÝ:
# NUMERICAL CONTINUOUS
# =========================================================

def xu_ly_num_cont(df, cols):

    # -----------------------------------------------------
    # THỐNG KÊ MÔ TẢ
    # -----------------------------------------------------

    summary = thong_ke_mo_ta_num_cont(

        df=df,
        cols=cols

    )

    display(summary)

    # -----------------------------------------------------
    # LOOP VẼ CHART
    # -----------------------------------------------------

    for col in cols:

        print(
            f'\n========== {col.upper()} =========='
        )

        ve_hist_boxplot(
            df=df,
            col=col
        )

# %%
# =========================================================
# HÀM TẠO SUMMARY
# DÀNH CHO:
# - NUMERICAL DISCRETE
# - CATEGORICAL NOMINAL
# - CATEGORICAL ORDINAL
# =========================================================

def tao_summary_categorical(df, col):

    summary = pd.DataFrame({

        'count': (

            df[col]

            .value_counts(
                dropna=False
            )

            .sort_index()

        ),

        'percent (%)': (

            df[col]

            .value_counts(
                normalize=True,
                dropna=False
            )

            .sort_index()

            * 100

        )

    })

    return summary.round(2)

# %%
# =========================================================
# HÀM VẼ:
# - COUNT
# - PERCENTAGE
# =========================================================

def ve_bar_chart_categorical(

    summary,
    title,

    rotation=90,

    figsize=(14,5),

    color_count='steelblue',
    color_percent='orange'

):

    fig, axes = plt.subplots(

        1,
        2,

        figsize=figsize

    )

    # -----------------------------------------------------
    # COUNT CHART
    # -----------------------------------------------------

    bars1 = axes[0].bar(

        summary.index.astype(str),

        summary['count'],

        color=color_count

    )

    axes[0].set_title(
        f'{title} - Count'
    )

    axes[0].tick_params(

        axis='x',
        rotation=rotation

    )

    # ----- Label count -----

    for b in bars1:

        axes[0].text(

            b.get_x() + b.get_width()/2,

            b.get_height(),

            int(b.get_height()),

            ha='center',
            va='bottom'

        )

    # -----------------------------------------------------
    # PERCENTAGE CHART
    # -----------------------------------------------------

    bars2 = axes[1].bar(

        summary.index.astype(str),

        summary['percent (%)'],

        color=color_percent

    )

    axes[1].set_title(
        f'{title} - Percentage (%)'
    )

    axes[1].tick_params(

        axis='x',
        rotation=rotation

    )

    # ----- Label percentage -----

    for b in bars2:

        axes[1].text(

            b.get_x() + b.get_width()/2,

            b.get_height(),

            f'{b.get_height():.2f}%',

            ha='center',
            va='bottom'

        )

    plt.tight_layout()

    plt.show()

# %%
# =========================================================
# HÀM XỬ LÝ:
# - NUMERICAL DISCRETE
# - CATEGORICAL NOMINAL
# - CATEGORICAL ORDINAL
# =========================================================

def xu_ly_categorical(
    df,
    cols,
    rotation=90
):

    for col in cols:

        print(
            f'\n========== {col.upper()} =========='
        )

        # -------------------------------------------------
        # TẠO SUMMARY
        # -------------------------------------------------

        summary = tao_summary_categorical(

            df=df,
            col=col

        )

        # -------------------------------------------------
        # HIỂN THỊ BẢNG
        # -------------------------------------------------

        display(summary)

        # -------------------------------------------------
        # VẼ CHART
        # -------------------------------------------------

        ve_bar_chart_categorical(

            summary=summary,

            title=col,

            rotation=rotation

        )

# %%
# =========================================================
# 1. NUMERICAL CONTINUOUS
# =========================================================

xu_ly_num_cont(
    df=df,
    cols=num_cont
)

# %%
# =========================================================
# 2. NUMERICAL DISCRETE
# =========================================================

xu_ly_categorical(
    df=df,
    cols=num_disc
)

# %%
# =========================================================
# 3. CATEGORICAL NOMINAL
# =========================================================

xu_ly_categorical(
    df=df,
    cols=cat_nom
)

# %%
# =========================================================
# 4. CATEGORICAL ORDINAL
# =========================================================

xu_ly_categorical(
    df=df,
    cols=cat_ord
)

# %%
# =========================================================
# XỬ LÝ OUTLIER
# SAU KHI ĐÃ QUAN SÁT HISTOGRAM + BOXPLOT
# =========================================================

# ----- Outlier emi_total -----

bad_emi_total = [

    12024818193,
    2556298158,
    1736127884,
    1144081658

]

df = df[
    ~df['emi_total'].isin(bad_emi_total)
].copy()

# ----- Outlier dti_total -----

bad_dti_total = [

    15031.02274,
    10225.19263,
    5787.092947

]

df = df[
    ~df['dti_total'].isin(bad_dti_total)
].copy()

# %%
# =========================================================
# DROP MISSING VALUE
# =========================================================

df = df.dropna().copy()

# %%
# =========================================================
# KIỂM TRA LẠI DATA SAU CLEAN
# =========================================================

print(df.shape)

display(
    kiem_tra_missing(df)
)

kiem_tra_duplicate(df)

# %%
# =========================================================
# XUẤT FILE DATA PROCESSED
# =========================================================

df.to_excel(
    'data_processed.xlsx',
    index=False
)